# Create Predict Price Model On Real Estate Data

## Connect to PostgreSQL

In [1]:
from pyspark.sql import SparkSession

jdbc_url = "jdbc:postgresql://postgres:5432/postgres"
table_name = "gold.gold_analytics_data"
connection_properties = {
    "user": "postgres",
    "password": "postgres"
}

## Import ML libraries

In [2]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml import Pipeline

## Load data

In [3]:
data = spark.read.jdbc(jdbc_url, table_name, properties=connection_properties)
data.printSchema()

root
 |-- sale_id: integer (nullable = true)
 |-- account_id: integer (nullable = true)
 |-- ad_id: integer (nullable = true)
 |-- area: string (nullable = true)
 |-- area_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- category_name: string (nullable = true)
 |-- latitude: decimal(10,6) (nullable = true)
 |-- longitude: decimal(10,6) (nullable = true)
 |-- location: string (nullable = true)
 |-- date: string (nullable = true)
 |-- price: long (nullable = true)
 |-- region_name: string (nullable = true)
 |-- rooms: integer (nullable = true)
 |-- size: decimal(10,2) (nullable = true)
 |-- type: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- price_per_sqm: decimal(33,11) (nullable = true)



## Data Preprocessing Pipeline

In [4]:
categorical_cols = ['area', 'area_name', 'category', 'category_name', 'location']
indexers = [StringIndexer(inputCol=col, outputCol=col + "_index").fit(data) for col in categorical_cols]
assembler = VectorAssembler(inputCols=['latitude', 'longitude', 'rooms', 'size', 'year', 'month', 'quarter'] + 
                                  [col + "_index" for col in categorical_cols], outputCol="raw_features")
scaler = StandardScaler(inputCol="raw_features", outputCol="features", withStd=True, withMean=True)
preprocessing_pipeline = Pipeline(stages=indexers + [assembler, scaler])
data_processed = preprocessing_pipeline.fit(data).transform(data)

## Split data into training and testing sets

In [5]:
train_data, test_data = data_processed.randomSplit([0.8, 0.2], seed=123)

## Initialize Gradient Boosted Trees Regressor

In [6]:
gbt = GBTRegressor(featuresCol="features", labelCol="price")

## Define parameter grid for hyperparameter tuning

In [7]:
param_grid = ParamGridBuilder() \
    .addGrid(gbt.maxDepth, [5, 10]) \
    .addGrid(gbt.maxIter, [50, 100]) \
    .build()

## Initialize CrossValidator

In [8]:
crossval = CrossValidator(estimator=gbt,
                          estimatorParamMaps=param_grid,
                          evaluator=RegressionEvaluator(labelCol="price", predictionCol="prediction", metricName="rmse"),
                          numFolds=3)

## Train the model

In [ ]:
model = crossval.fit(train_data)

## Make predictions

In [10]:
predictions = model.transform(test_data)

## Evaluate the model

In [12]:
evaluator = RegressionEvaluator(labelCol="price", predictionCol="prediction", metricName="rmse")
rmse = evaluator.evaluate(predictions)
print("Root Mean Squared Error (RMSE) on test data = %g" % rmse)

Root Mean Squared Error (RMSE) on test data = 3.30854e+10


## Bruh, this model is so bad, I dont know why @@